In [15]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import re
from urllib.parse import urljoin, parse_qs, urlparse
import random

class GoogleScholarCitationScraper:
    def __init__(self):
        self.session = requests.Session()
        # Use a realistic user agent
        self.session.headers.update({
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
        })
        self.base_url = "https://scholar.google.com"
        
    def get_page(self, url, delay_range=(5, 12), max_retries=3):
        """Get a page with random delay and retry logic to avoid rate limiting"""
        for attempt in range(max_retries):
            # Longer delays to avoid rate limiting
            delay = random.uniform(*delay_range)
            print(f"  Waiting {delay:.1f} seconds before request...")
            time.sleep(delay)
            
            try:
                response = self.session.get(url, timeout=30)
                response.raise_for_status()
                
                # Check if we got a CAPTCHA or blocked page
                if "captcha" in response.text.lower() or "blocked" in response.text.lower():
                    print(f"  CAPTCHA/Block detected on attempt {attempt + 1}")
                    if attempt < max_retries - 1:
                        print(f"  Waiting longer before retry...")
                        time.sleep(random.uniform(30, 60))  # Wait 30-60 seconds
                        continue
                    else:
                        print("  Max retries reached, may be blocked")
                        return None
                        
                return response
                
            except requests.RequestException as e:
                print(f"  Error on attempt {attempt + 1}: {e}")
                if attempt < max_retries - 1:
                    wait_time = random.uniform(10, 30)
                    print(f"  Retrying in {wait_time:.1f} seconds...")
                    time.sleep(wait_time)
                else:
                    print(f"  Max retries reached for {url}")
                    return None
        
        return None
    
    def extract_year_from_text(self, text):
        """Extract publication year from text"""
        year_patterns = [
            r'\b(20[0-2][0-9]|19[5-9][0-9])\b',
            r'\((20[0-2][0-9]|19[5-9][0-9])\)',
            r'- (20[0-2][0-9]|19[5-9][0-9])\b'
        ]
        
        years = []
        for pattern in year_patterns:
            matches = re.findall(pattern, text)
            years.extend([int(match) for match in matches if isinstance(match, str)])
        
        if not years:
            return "Year not found"
            
        valid_years = [y for y in years if 1950 <= y <= 2024]
        if not valid_years:
            return "Year not found"
            
        return max(valid_years)
    
    def find_cited_by_url(self, main_paper_url):
        """Find the 'Cited by' URL from the main paper page"""
        print(f"Getting main paper page: {main_paper_url}")
        response = self.get_page(main_paper_url)
        
        if not response:
            print("Could not access main paper page")
            return None
            
        soup = BeautifulSoup(response.content, 'html.parser')
        
        # Look for "Cited by" link
        cited_by_patterns = [
            'a[href*="cites="]',
            'a[href*="cites"]',
            'a:contains("Cited by")',
            'a:contains("cited by")'
        ]
        
        cited_by_url = None
        for pattern in cited_by_patterns:
            try:
                cited_by_elem = soup.select_one(pattern)
                if cited_by_elem and 'cites' in cited_by_elem.get('href', ''):
                    href = cited_by_elem.get('href')
                    if href.startswith('http'):
                        cited_by_url = href
                    else:
                        cited_by_url = urljoin(self.base_url, href)
                    
                    # Check if this looks like a citation link
                    cite_text = cited_by_elem.get_text().lower()
                    if 'cited by' in cite_text or 'cites' in href:
                        print(f"Found 'Cited by' link: {cited_by_url}")
                        return cited_by_url
            except:
                continue
        
        # Alternative approach: look for the citation count and build URL manually
        print("Trying alternative approach to find citations...")
        
        # Look for citation count in the page text
        page_text = soup.get_text()
        cite_matches = re.findall(r'(?:Cited by|Citations?:?\s*)(\d+)', page_text, re.IGNORECASE)
        
        if cite_matches:
            citation_count = max([int(match) for match in cite_matches])
            print(f"Found citation count: {citation_count}")
            
            # Try to extract paper ID from the original URL
            if 'cluster=' in main_paper_url:
                cluster_id = re.search(r'cluster=(\d+)', main_paper_url)
                if cluster_id:
                    cluster_id = cluster_id.group(1)
                    cited_by_url = f"https://scholar.google.com/scholar?cites={cluster_id}&as_sdt=2005&sciodt=0,5&hl=en"
                    print(f"Constructed citations URL: {cited_by_url}")
                    return cited_by_url
        
        return None
    
    def extract_authors_and_venue(self, citation):
        """Extract authors and publication venue information"""
        author_elem = citation.select_one('.gs_a')
        if author_elem:
            author_text = author_elem.get_text().strip()
            parts = author_text.split(' - ')
            if len(parts) >= 2:
                authors = parts[0].strip()
                venue = ' - '.join(parts[1:]).strip()
            else:
                authors = author_text
                venue = "Venue not specified"
        else:
            authors = "Authors not found"
            venue = "Venue not found"
        
        return authors, venue
    
    def scrape_all_citations(self, citations_url):
        """Scrape all citing papers from the citations URL with enhanced rate limiting"""
        all_citations = []
        start = 0
        page_size = 10
        max_pages = 200
        consecutive_empty_pages = 0
        max_consecutive_empty = 8  # Increased tolerance
        
        print(f"Starting to scrape all citations from: {citations_url}")
        print("Using enhanced rate limiting to avoid blocks...")
        
        while consecutive_empty_pages < max_consecutive_empty and start < (max_pages * page_size):
            # Construct URL for current page
            if start == 0:
                current_url = citations_url
            else:
                if '&start=' in citations_url:
                    current_url = re.sub(r'&start=\d+', f'&start={start}', citations_url)
                else:
                    current_url = f"{citations_url}&start={start}"
            
            print(f"\nScraping citations page {start//page_size + 1} (results {start}-{start+page_size-1})...")
            
            # Add extra delay every 10 pages
            if (start // page_size) % 10 == 0 and start > 0:
                extra_delay = random.uniform(30, 60)
                print(f"  Taking extended break ({extra_delay:.1f}s) to avoid rate limits...")
                time.sleep(extra_delay)
            
            response = self.get_page(current_url)
            
            if not response:
                print("  Failed to get response after retries")
                consecutive_empty_pages += 1
                
                # If we've been getting failures, take a longer break
                if consecutive_empty_pages >= 3:
                    long_delay = random.uniform(120, 300)  # 2-5 minute break
                    print(f"  Multiple failures detected. Taking long break ({long_delay/60:.1f} minutes)...")
                    time.sleep(long_delay)
                
                start += page_size
                continue
                
            soup = BeautifulSoup(response.content, 'html.parser')
            
            # Check for rate limiting indicators
            page_text = response.text.lower()
            if any(indicator in page_text for indicator in ['captcha', 'unusual traffic', 'blocked', 'try again later']):
                print("  Rate limiting detected in page content!")
                long_delay = random.uniform(300, 600)  # 5-10 minute break
                print(f"  Taking extended break ({long_delay/60:.1f} minutes)...")
                time.sleep(long_delay)
                continue
            
            # Find citation entries
            citations = (soup.find_all('div', class_='gs_r gs_or gs_scl') or 
                        soup.find_all('div', class_='gs_ri') or
                        soup.find_all('div', class_='gs_r'))
            
            if not citations:
                citations = soup.find_all('div', {'class': re.compile(r'gs_r')})
            
            if not citations:
                consecutive_empty_pages += 1
                print(f"  No citations found on page {start//page_size + 1}")
                print(f"  Consecutive empty pages: {consecutive_empty_pages}/{max_consecutive_empty}")
                
                # Check if this might be the end or rate limiting
                if consecutive_empty_pages >= 3:
                    print("  Multiple empty pages - might be rate limited or end of results")
                    if consecutive_empty_pages >= 5:
                        # Try a different approach - check if we can access an earlier page
                        test_url = citations_url if start > 50 else f"{citations_url}&start=10"
                        print(f"  Testing connectivity with: {test_url}")
                        test_response = self.get_page(test_url)
                        if not test_response:
                            print("  Connection test failed - likely blocked")
                            break
                
                start += page_size
                continue
            else:
                consecutive_empty_pages = 0
            
            citations_found_on_page = 0
            
            for citation in citations:
                try:
                    # Extract title
                    title = "Title not found"
                    paper_url = None
                    
                    title_selectors = [
                        ('h3.gs_rt a', True),
                        ('h3.gs_rt', False),
                        ('.gs_rt a', True),
                        ('.gs_rt', False),
                        ('h3 a', True),
                        ('h3', False)
                    ]
                    
                    for selector, has_link in title_selectors:
                        title_elem = citation.select_one(selector)
                        if title_elem:
                            title = title_elem.get_text().strip()
                            if has_link and title_elem.get('href'):
                                href = title_elem.get('href')
                                if href.startswith('http'):
                                    paper_url = href
                                elif href.startswith('/'):
                                    paper_url = urljoin(self.base_url, href)
                            break
                    
                    if title == "Title not found" or len(title.strip()) < 5:
                        continue
                    
                    # Extract abstract
                    abstract = "Abstract not available"
                    abstract_selectors = ['.gs_rs', '.gs_a ~ div', 'div.gs_rs']
                    
                    for selector in abstract_selectors:
                        abstract_elem = citation.select_one(selector)
                        if abstract_elem:
                            abstract_text = abstract_elem.get_text().strip()
                            if abstract_text and len(abstract_text) > 20:
                                abstract = abstract_text
                                break
                    
                    # Extract year
                    citation_text = citation.get_text()
                    year = self.extract_year_from_text(citation_text)
                    
                    # Extract authors and venue
                    authors, venue = self.extract_authors_and_venue(citation)
                    
                    # Extract citation count
                    citation_count = "N/A"
                    cite_elem = citation.select_one('a[href*="cites"]')
                    if cite_elem:
                        cite_text = cite_elem.get_text()
                        cite_match = re.search(r'Cited by (\d+)', cite_text)
                        if cite_match:
                            citation_count = int(cite_match.group(1))
                    
                    citation_data = {
                        'Title': title,
                        'Authors': authors,
                        'Venue': venue,
                        'Year': year,
                        'Abstract': abstract,
                        'Citations': citation_count,
                        'URL': paper_url if paper_url else 'N/A'
                    }
                    
                    all_citations.append(citation_data)
                    citations_found_on_page += 1
                    
                    if len(all_citations) % 25 == 0:  # More frequent progress updates
                        print(f"    Progress: {len(all_citations)} citations scraped...")
                        
                except Exception as e:
                    print(f"    Error processing citation: {e}")
                    continue
            
            print(f"  Found {citations_found_on_page} citations on page {start//page_size + 1}")
            print(f"  Total collected so far: {len(all_citations)}")
            
            start += page_size
            
            # Progress milestone updates
            if len(all_citations) > 0 and len(all_citations) % 100 == 0:
                print(f"\n*** MILESTONE: {len(all_citations)} citations collected! ***")
                # Save intermediate results
                if len(all_citations) >= 100:
                    temp_df = pd.DataFrame(all_citations)
                    temp_filename = f'citations_partial_{len(all_citations)}.csv'
                    temp_df.to_csv(temp_filename, index=False, encoding='utf-8')
                    print(f"*** Intermediate results saved to {temp_filename} ***\n")
        
        print(f"\nScraping completed! Found {len(all_citations)} total citations")
        return all_citations
    
    def save_to_table(self, citations, filename='all_citations.csv'):
        """Save citations to CSV file and display summary"""
        if not citations:
            print("No citations found to save.")
            return None
        
        df = pd.DataFrame(citations)
        df.to_csv(filename, index=False, encoding='utf-8')
        print(f"Saved {len(citations)} citations to {filename}")
        
        # Display summary statistics
        print(f"\n{'='*100}")
        print("CITATION ANALYSIS SUMMARY")
        print(f"{'='*100}")
        print(f"Total citing papers found: {len(citations)}")
        
        # Year distribution
        year_counts = {}
        for paper in citations:
            year = paper['Year']
            if year != "Year not found":
                year_counts[year] = year_counts.get(year, 0) + 1
        
        if year_counts:
            print(f"\nYear distribution of citing papers:")
            for year in sorted(year_counts.keys(), reverse=True):
                print(f"  {year}: {year_counts[year]} papers")
        
        # Show first 10 citations as preview
        print(f"\n{'='*100}")
        print("PREVIEW: FIRST 10 CITING PAPERS")
        print(f"{'='*100}")
        
        for i, paper in enumerate(citations[:10], 1):
            print(f"\n{i}. Title: {paper['Title']}")
            print(f"   Authors: {paper['Authors']}")
            print(f"   Venue: {paper['Venue']}")
            print(f"   Year: {paper['Year']}")
            print(f"   Citations: {paper['Citations']}")
            
            abstract = paper['Abstract']
            abstract_preview = abstract[:300] + '...' if len(abstract) > 300 else abstract
            print(f"   Abstract: {abstract_preview}")
            print(f"   URL: {paper['URL']}")
            print("-" * 80)
        
        if len(citations) > 10:
            print(f"\n... and {len(citations) - 10} more citations (see CSV file for complete list)")
        
        return df

def main():
    # The main paper URL you provided
    main_paper_url = "https://scholar.google.com/scholar?cluster=2469397274690356930&hl=en&as_sdt=2005&sciodt=0,5&as_ylo=2024"
    
    scraper = GoogleScholarCitationScraper()
    
    try:
        print(f"{'='*80}")
        print("GOOGLE SCHOLAR CITATION SCRAPER")
        print(f"{'='*80}")
        print(f"Main paper URL: {main_paper_url}")
        print()
        
        # Step 1: Find the citations URL
        citations_url = scraper.find_cited_by_url(main_paper_url)
        
        if not citations_url:
            print("Could not find the 'Cited by' link. Trying manual construction...")
            # Manual construction based on cluster ID
            cluster_match = re.search(r'cluster=(\d+)', main_paper_url)
            if cluster_match:
                cluster_id = cluster_match.group(1)
                citations_url = f"https://scholar.google.com/scholar?cites={cluster_id}&as_sdt=2005&sciodt=0,5&hl=en"
                print(f"Constructed citations URL: {citations_url}")
            else:
                print("Could not construct citations URL. Please check the paper URL.")
                return
        
        # Step 2: Scrape all citations
        citations = scraper.scrape_all_citations(citations_url)
        
        if citations:
            # Step 3: Save and display results
            df = scraper.save_to_table(citations)
            print(f"\n*** SCRAPING COMPLETED SUCCESSFULLY! ***")
            print(f"Found {len(citations)} citing papers")
            print(f"Results saved to 'all_citations.csv'")
        else:
            print("No citations were found.")
                
    except KeyboardInterrupt:
        print("\nScraping interrupted by user.")
    except Exception as e:
        print(f"An error occurred: {e}")
        import traceback
        traceback.print_exc()

if __name__ == "__main__":
    main()

GOOGLE SCHOLAR CITATION SCRAPER
Main paper URL: https://scholar.google.com/scholar?cluster=2469397274690356930&hl=en&as_sdt=2005&sciodt=0,5&as_ylo=2024

Getting main paper page: https://scholar.google.com/scholar?cluster=2469397274690356930&hl=en&as_sdt=2005&sciodt=0,5&as_ylo=2024
  Waiting 6.1 seconds before request...
  CAPTCHA/Block detected on attempt 1
  Waiting longer before retry...

Scraping interrupted by user.


In [13]:
df = pd.read_csv('all_papers.csv')

In [14]:
df

,Title,Authors,Venue,Year,Abstract,Citations,URL
0,Deepseek-r1: Incentivizing reasoning capabilit...,"D Guo, D Yang, H Zhang, J Song, R Zhang… - arX...",arxiv.org,Year not found,We introduce our first-generation reasoning mo...,1436,https://arxiv.org/abs/2501.12948
